### Simple CNN & BP Implementation (2025)

- Implemented a convolutional neural network from scratch in **PyTorch**, including 2D convolution, ReLU, batch normalization, and their corresponding gradient operators
- Implemented back-propagation manually and verified gradients using **finite-difference gradient checking** against PyTorch autograd
- Built a two-layer CNN for **MNIST classification** and implemented gradient computation for convolutional filters and the linear output layer\


In [29]:
# Imports
import pdb
import torch
import numpy as np
from torch import nn
from math import pi
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import datasets, transforms, utils

torch.set_default_dtype(torch.float64)

## load MNIST images
B = 5 # batch size
train_set = datasets.MNIST('./data',
                            train=True,
                            download=True,
                            transform=transforms.ToTensor())

loader = torch.utils.data.DataLoader(train_set, batch_size=B)

## load a batch of MNIST images as a PyTorch tensor (shape: B x C x H x W)
# B: batch size
# C: number of channels
# H: height of images
# W: width of images
img, label = next(iter(loader)) # img shape: B x C x H x W, label shape: B X 1

## create a random filter (shape: D x C x K x K)
K = 3 # kernel size
P = 1 # padding size
C = 1 # channel size
D = 2 # number of filters
filter = torch.randn(D, C, K, K) # filter shape: D x C x K x K

In [ ]:
def conv2d_im2col(img, filter, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    ### Fill in this function ###
    # Args:
    #   img: images, shape B x C x H x W
    #   filter: filters, shape D x C x K x K
    #   channel_size: number of channels, scalar (C)
    #   num_filters: number of filters, scalar (D)
    #   kernel_size: kernel size, scalar (K)
    #   stride: stride size, scalar
    #   padding: padding size, scalar
    #
    # Returns:
    #   out: convoluted images, shape B x D x H x W
    
    
    # get 4d tensors dimensions
    D, C, K, K = filter.size() 
    B, C, H, W = img.size()

    # Get our output sizes
    # Use formula from HW2:
    # "[(W−K+2P)/S]+1"
    W_out = (W - K + 2*padding)//stride + 1 
    H_out = (H - K + 2*padding)//stride + 1 
    result = torch.zeros((B, D, H_out, W_out)) 

    # Zero padding by using 2P + dim and then putting in middle of zeros tensor (H and W are padded)
    H_padded = H + 2*padding
    W_padded = W + 2*padding
    x_zero_padded = torch.zeros((B, C, H_padded, W_padded))
    # : selects all
    x_zero_padded[:, :, padding:H_padded-padding, padding:W_padded-padding] = img

    # Convolution through matmul, reshape to D (1d and follow construction)
    h_window = filter.reshape(D, -1)

    # Convolute through multiplicaiton view
    for i in range(H_out):
        for j in range(W_out):
            x_region = x_zero_padded[:, :, i*stride : i*stride+K , j*stride : j*stride+K]
            x_flat_b= x_region.reshape(B, -1) 
            # x flattened to b X h_window before matmul
            conv = torch.matmul(x_flat_b, h_window.T) 
            result[:, :, i, j] = conv.reshape(B, D) 

    return result


def conv2d_filter2row(img, filter, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    ### Fill in this function ###
    # Args:
    #   img: images, shape B x C x H x W
    #   filter: filters, shape D x C x K x K
    #   channel_size: number of channels, scalar (C)
    #   num_filters: number of filters, scalar (D)
    #   kernel_size: kernel size, scalar (K)
    #   stride: stride size, scalar
    #   padding: padding size, scalar
    #
    # Returns:
    #   out: convoluted images, shape B x D x H x W
        ### Padding code, size code is same as above implementation:


    # Repeat above process: (img2col)
    # Init sizes
    D, _, K, _ = filter.size() 
    B, C, H, W = img.size()
    
    # Output shape
    H_out = (H-K+2*padding)//stride + 1 
    W_out = (W-K+2*padding)//stride + 1 
    result = torch.zeros((B, D, H_out, W_out)) 

    # Pad
    H_padded = H + 2*padding
    W_padded = W + 2*padding
    x_zero_padded = torch.zeros((B, C, H_padded, W_padded))
    x_zero_padded[:, :, padding:H_padded-padding, padding:W_padded-padding] = img

    h_window = filter.reshape(D, -1)
    
    # run loops to matmul
    for ii in range(H_out):
        for jj in range(W_out):
            x_region = x_zero_padded[:, :, ii*stride : ii*stride+K , jj*stride : jj*stride+K]
            x_flattened_b= x_region.reshape(B, -1) 
            # Now different: Note how order is different now since it's filter2row
            # Matmul is flipped, and transposed (.T)
            conv = torch.matmul(h_window, x_flattened_b.T) 
            result[:, :, ii, jj] = conv.T.reshape(B, D) 

    return result


def unit_test_conv2d(img, filter, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    # call your implemented "im2col" conv2D
    y_im2col = conv2d_im2col(img, filter, channel_size=channel_size, num_filters=num_filters, kernel_size=kernel_size, stride=stride, padding=padding)

    # ground truth conv2D
    y_gt = F.conv2d(img, filter, stride=stride, padding=padding)

    diff = (y_im2col - y_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of conv2d_im2col is correct!")
    else:
        print("Your implementation of conv2d_im2col is wrong!")

    # call your implemented "im2col" conv2D
    y_filter2row = conv2d_filter2row(img, filter, channel_size=channel_size, num_filters=num_filters, kernel_size=kernel_size, stride=stride, padding=padding)

    diff = (y_filter2row - y_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of conv2d_filter2row is correct!")
    else:
        print("Your implementation of conv2d_filter2row is wrong!")


unit_test_conv2d(img, filter, channel_size=C, num_filters=D, kernel_size=K, stride=1, padding=P)

Your implementation of conv2d_im2col is correct!
Your implementation of conv2d_filter2row is correct!


In [ ]:
def grad_conv2d(img, filter, out, grad_out, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    ### Fill in this function ###
    # Args:
    #   img: images, shape B x C x H x W
    #   filter: filters, shape D x C x K x K
    #   out: convoluted images, shape B x D x H x W
    #   grad_out: gradient w.r.t. output, shape B x D x H x W
    #   channel_size: number of channels, scalar (C)
    #   num_filters: number of filters, scalar (D)
    #   kernel_size: kernel size, scalar (K)
    #   stride: stride size, scalar
    #   padding: padding size, scalar
    #
    # Returns:
    #   grad_img: gradient w.r.t. img, shape B x C x H x W
    #   grad_filer: gradient w.r.t. filter, shape D x C x K x K


    # Get shapes 
    B, C, H, W = img.shape  
    D, C, K, K = filter.shape  

    # Pad (use above method)
    H_padded = H + 2*padding
    W_padded = W + 2*padding
    x_padded = torch.zeros((B, C, H_padded, W_padded))
    x_padded[:, :, padding:H_padded-padding, padding:W_padded-padding] = img

    # Compute gradients:

    # Gradient filter
    grad_filter = torch.zeros_like(filter)
    for b in range(B):
        for d in range(D):
            for h in range(H):
                for w in range(W):
                    # Convolve grad_out and x_pad, elementwise mult
                    conv = (grad_out[b, d, h, w] * x_padded[b, :, h:h+K, w:w+K])
                    # filter is D C K K
                    grad_filter[d, :, :, :] = grad_filter[d, :, :, :] + conv
    # gradient image
    grad_img = torch.zeros_like(x_padded)
    for b in range(B):
        for d in range(D):
            for h in range(H):
                for w in range(W):
                    # Convolve filter with each gradient vector 
                    # filter is D C K K (height and width = K)
                    conv = (grad_out[b, d, h, w] * filter[d, :, :, :])
                    grad_img[b, :, h:h+K, w:w+K] = grad_img[b, :, h:h+K, w:w+K] + conv


    # Strip padding
    grad_img = grad_img[:, :, padding:H+padding, padding:W+padding]
    
    return grad_img, grad_filter



def unit_test_grad_conv2d(img, filter, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    filter.requires_grad = True
    img.requires_grad = True

    ### ground truth conv2D
    img_out = F.conv2d(img, filter, stride=stride, padding=padding)

    # create a random vector v
    v = torch.randn_like(img_out)

    # call your implemented "grad_conv2d" function
    grad_img, grad_filter = grad_conv2d(img, filter, img_out, v, channel_size=channel_size, num_filters=num_filters, kernel_size=kernel_size, stride=stride, padding=padding)

    # compute ground-truth gradients
    grad_img_gt = torch.autograd.grad(img_out, img, grad_outputs=v, retain_graph=True)[0]
    grad_filter_gt = torch.autograd.grad(img_out, filter, grad_outputs=v, retain_graph=True)[0]
    #pdb.set_trace()

    diff = (grad_img - grad_img_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_img is correct!")
    else:
        print("Your implementation of grad_img is wrong!")

    diff = (grad_filter - grad_filter_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_filter is correct!")
    else:
        print("Your implementation of grad_filter is wrong!")

unit_test_grad_conv2d(img, filter, channel_size=C, num_filters=D, kernel_size=K, stride=1, padding=P)

Your implementation of grad_img is correct!
Your implementation of grad_filter is correct!


In [ ]:
def grad_checker(img, filter, conv, grad_out, epsilon=1.0e-5, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    ### Fill in this function ###
    # Args:
    #   img: images, shape B x C x H x W
    #   filter: filters, shape D x C x K x K
    #   conv: convolution function
    #   out: convoluted images, shape B x D x H x W
    #   grad_out: gradient w.r.t. output, shape B x D x H x W
    #   channel_size: number of channels, scalar (C)
    #   num_filters: number of filters, scalar (D)
    #   kernel_size: kernel size, scalar (K)
    #   stride: stride size, scalar
    #   padding: padding size, scalar
    #
    # Returns:
    #   grad_img: gradient w.r.t. img, shape B x C x H x W
    #   grad_filer: gradient w.r.t. filter, shape D x C x K x K
  

    # Finite difference approximation:
    # D,C,K,K
    grad_filter = torch.zeros_like(filter)
    for i in range(filter.shape[0]): 
        for j in range(filter.shape[1]): 
            for k in range(filter.shape[2]): 
                for z in range(filter.shape[3]): 
    
                    # Clone tensor for modify
                    filt_plus = filter.clone()
                    filt_minus = filter.clone()
                    
                    # 1. Add epsilon every loop and convolute (we can use conv(), we passed in)
                    filt_plus[i,j,k,z] = filt_plus[i,j,k,z]+ epsilon
                    filt_minus[i,j,k,z] = filt_minus[i,j,k,z] - epsilon

                    filt_plus_conv = conv(img, filt_plus, stride=stride, padding=padding) 
                    filt_minus_conv = conv(img, filt_minus, stride=stride, padding=padding) 
                    
                    # 2. matmul with grad
                    filt_plus_product = torch.mul(filt_plus_conv, grad_out)
                    filt_minus_product = torch.mul(filt_minus_conv, grad_out)

                    # 3. add loss on summed dimension
                    filt_plus_loss = torch.sum(filt_plus_product)
                    filt_minus_loss = torch.sum(filt_minus_product)

                    # Do the formula ~(h + epsilon)-l( h - epsilon)/(2*epsilon)
                    grad_filter[i,j,k,z] = (filt_plus_loss-filt_minus_loss) / (2*epsilon)

    # Finite difference approximation:
    # B,C,H,W
    # copy above implementation.. on BCHW instead
    grad_img = torch.zeros_like(img)
    for b in range(img.shape[0]): 
        for c in range(img.shape[1]): 
            for h in range(img.shape[2]): 
                for w in range(img.shape[3]): 

                    # Clone tensor for modify
                    img_plus = img.clone()
                    img_minus = img.clone()

                    # 1. Add epsilon every loop and conv
                    img_plus[b, c, h, w] = img_plus[b, c, h, w] + epsilon
                    img_minus[b, c, h, w] = img_minus[b, c, h, w] - epsilon

                    img_plus_conv = conv(img_plus, filter, stride=stride, padding=padding) 
                    img_minus_conv = conv(img_minus, filter, stride=stride, padding=padding) 

                    # 2. matmul with grad
                    img_plus_product = torch.mul(img_plus_conv, grad_out)
                    img_minus_product = torch.mul(img_minus_conv, grad_out)

                    # 3. add loss per dimension
                    img_plus_loss = torch.sum(img_plus_product)
                    img_minus_loss = torch.sum(img_minus_product)

                    # Do the formula ~(h + epsilon)-l( h - epsilon)/(2*epsilon)
                    grad_img[b, c, h, w] = (img_plus_loss - img_minus_loss) / (2 * epsilon)

    # return outputs
    return grad_img, grad_filter


def unit_test_grad_checker(img, filter, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    epsilon = 1.0e-5
    filter.requires_grad = True
    img.requires_grad = True

    ### ground truth conv2D
    img_out = F.conv2d(img, filter, stride=stride, padding=padding)

    # create a random vector v
    v = torch.randn_like(img_out)

    # call your implemented "grad_checker" function
    grad_img, grad_filter = grad_checker(img, filter, F.conv2d, v, epsilon=epsilon, channel_size=channel_size, num_filters=num_filters, kernel_size=kernel_size, stride=stride, padding=padding)

    # compute ground-truth gradients
    grad_img_gt = torch.autograd.grad(img_out, img, grad_outputs=v, retain_graph=True)[0]
    grad_filter_gt = torch.autograd.grad(img_out, filter, grad_outputs=v, retain_graph=True)[0]
    #pdb.set_trace()

    diff = (grad_img - grad_img_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_img is correct!")
    else:
        print("Your implementation of grad_img is wrong!")

    diff = (grad_filter - grad_filter_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_filter is correct!")
    else:
        print("Your implementation of grad_filter is wrong!")

unit_test_grad_checker(img, filter, channel_size=C, num_filters=D, kernel_size=K, stride=1, padding=P)

Your implementation of grad_img is correct!
Your implementation of grad_filter is correct!


In [ ]:
def func_relu(x):
    ### Fill in this function ###
    # Args:
    #   x: input, shape B x C x H x W
    #
    # Returns:
    #   y: output, shape B x C x H x W


    # ReLU: f(x) = max(x,0)
    zeros = torch.zeros_like(x)
    result = torch.maximum(x,zeros) 

    return result


def grad_relu(x, y, grad_out):
    ### Fill in this function ###
    # Args:
    #   x: input, shape B x C x H x W
    #   y: output, shape B x C x H x W
    #   grad_out: gradient w.r.t. output y, shape B x D x H x W
    #
    # Returns:
    #   grad_x: gradient w.r.t. x, shape B x C x H x W


    # Do gradient of relu
    # https://stackoverflow.com/questions/42042561/relu-derivative-in-backpropagation
    zeros = torch.zeros_like(x)
    sign = torch.sign(x)
    result = grad_out * torch.maximum(zeros, sign)

    return result


def unit_test_relu(x):
    x.requires_grad = True

    # call your implemented "func_relu" function
    y = func_relu(x)

    # ground truth ReLU
    y_gt = F.relu(x)

    diff = (y - y_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of func_relu is correct!")
    else:
        print("Your implementation of func_relu is wrong!")

    # create a random vector v
    v = torch.randn_like(y)

    # call your implemented "grad_relu" function
    grad_x = grad_relu(x, y, v)

    # compute ground-truth gradients
    grad_x_gt = torch.autograd.grad(y_gt, x, grad_outputs=v, retain_graph=True)[0]

    diff = (grad_x - grad_x_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_relu is correct!")
    else:
        print("Your implementation of grad_relu is wrong!")

unit_test_relu(torch.randn_like(img))

Your implementation of func_relu is correct!
Your implementation of grad_relu is correct!


In [ ]:
def func_batch_norm(x, epsilon=1.0e-5):
    ### Fill in this function ###
    # Args:
    #   x: input, shape B x C x H x W
    #   epsilon: constant, scalar
    #
    # Returns:
    #   y: output, shape B x C x H x W


    # BHW = Dim 0,2,3 from (0,1,2,3)
    dims = (0, 2, 3)
    # Mean[c]
    mean_c = torch.mean( x , dims , keepdim=True )
    # Var[c]
    var_c = torch.pow( (x - mean_c) , 2)
    var = torch.mean( var_c, dims , keepdim=True)

    # BN formula
    result = (x - mean_c) / torch.sqrt(var + epsilon)

    return result

def grad_batch_norm(x, y, grad_out, epsilon=1.0e-5):
    ### Fill in this function ###
    # Args:
    #   x: input, shape B x C x H x W
    #   y: output, shape B x C x H x W
    #   grad_out: gradient w.r.t. output y, shape B x D x H x W
    #   epsilon: constant, scalar
    #
    # Returns:
    #   grad_x: gradient w.r.t. x, shape B x C x H x W


    # Gradient BN formulae
    # https://kevinzakka.github.io/2016/09/14/batch_normalization/
    # BHW 
    dims = (0, 2, 3)
    # var & sqrt
    var_c = torch.var(x, dims, unbiased=False, keepdim=True)
    sqrt_var_c = torch.sqrt(var_c + epsilon)
    sqrt_var_x_square = torch.pow(sqrt_var_c, 2)
    mean_square_sqrt_var_x = torch.mean(sqrt_var_x_square, dims, keepdim=True)
    # -mean
    input_minus_mean = x - torch.mean(x, dims, keepdim=True)
    mean_grad_input_minus_mean = torch.mean(torch.mul(grad_out, input_minus_mean), dims, keepdim=True)
    # norm
    grad_norm = grad_out - torch.mean(grad_out, dims, keepdim=True)
    # gradient 
    result = (grad_norm - input_minus_mean / mean_square_sqrt_var_x * mean_grad_input_minus_mean ) / sqrt_var_c

    return result


def unit_test_batch_norm(x):
    x.requires_grad = True
    epsilon = 1e-5

    # call your implemented "func_batch_norm" function
    y = func_batch_norm(x, epsilon=epsilon)

    # ground truth ReLU
    BN_gt = nn.BatchNorm2d(x.shape[1], eps=epsilon, momentum=1.0, affine=False, track_running_stats=False)
    y_gt = BN_gt(x)

    diff = (y - y_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of func_batch_norm is correct!")
    else:
        print("Your implementation of func_batch_norm is wrong!")

    # create a random vector v
    v = torch.randn_like(y)

    # call your implemented "grad_batch_norm" function
    grad_x = grad_batch_norm(x, y, v, epsilon=epsilon)

    # compute ground-truth gradients
    grad_x_gt = torch.autograd.grad(y_gt, x, grad_outputs=v, retain_graph=True)[0]

    diff = (grad_x - grad_x_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_batch_norm is correct!")
    else:
        print("Your implementation of grad_batch_norm is wrong!")

unit_test_batch_norm(torch.randn_like(img))

Your implementation of func_batch_norm is correct!
Your implementation of grad_batch_norm is correct!


In [ ]:
def CNN(img, filter_1, filter_2, weight, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    ### Fill in this function ###
    # Args:
    #   img: images, shape B x C x H x W
    #   filter_1: filters at 1st layer, shape D x C x K x K
    #   filter_2: filters at 2nd layer, shape D x C x K x K
    #   weight: weights of linear readout layer, shape ? x 10
    #   channel_size: number of channels, scalar (C)
    #   num_filters: number of filters, scalar (D)
    #   kernel_size: kernel size, scalar (K)
    #   stride: stride size, scalar
    #   padding: padding size, scalar
    #
    # Returns:
    #   out: logits, shape B x 10


    B, C, H, W = img.shape

    # CONV -> BN -> RELU -> CONV -> BN -> RELU
    # First layer
    conv1 = conv2d_im2col(img, filter_1, channel_size, num_filters, kernel_size, stride, padding)
    bn1 = func_batch_norm(conv1)
    relu1 = func_relu(bn1)

    # Second layer 
    conv2 = conv2d_im2col(relu1, filter_2, num_filters, num_filters, kernel_size, stride, padding)
    bn2 = func_batch_norm(conv2)
    relu2 = func_relu(bn2)

    # Flatten to linear it to B x 10
    B, D, H_out, W_out = relu2.shape
    flattened = relu2.view(B, -1)  # B x (H,W)
    logits = torch.matmul(flattened, weight)  

    return logits


def grad_CNN(img, filter_1, filter_2, weight, grad_loss, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    ### Fill in this function ###
    # Args:
    #   img: images, shape B x C x H x W
    #   filter_1: filters at 1st layer, shape D x C x K x K
    #   filter_2: filters at 2nd layer, shape D x C x K x K
    #   weight: weights of linear readout layer, shape ? x 10
    #   grad_loss: gradient of loss w.r.t. logits, shape B x 10
    #   channel_size: number of channels, scalar (C)
    #   num_filters: number of filters, scalar (D)
    #   kernel_size: kernel size, scalar (K)
    #   stride: stride size, scalar
    #   padding: padding size, scalar
    #
    # Returns:
    #   grad_filter_1: filters, shape D x C x K x K
    #   grad_filter_2: filters, shape D x C x K x K
    #   grad_weight: weight, shape ? x 10


    # CNN
    # Forward pass to get all the values
    # CONV -> BN -> RELU -> CONV -> BN -> RELU
    conv1 = conv2d_im2col(img, filter_1, channel_size, num_filters, kernel_size, stride, padding)
    bn1 = func_batch_norm(conv1)
    relu1 = func_relu(bn1)

    conv2 = conv2d_im2col(relu1, filter_2, num_filters, num_filters, kernel_size, stride, padding)
    bn2 = func_batch_norm(conv2)
    relu2 = func_relu(bn2)

    B, D, H_out, W_out = relu2.shape
    flattened = relu2.view(B, -1)

    # Grad CNN
    # Do BP (backwards)
    # CONV -> BN -> RELU -> CONV -> BN -> RELU but in reverse
    grad_flat = torch.matmul(grad_loss, weight.T)
    grad_flat = grad_flat.view(B, D, H_out, W_out)

    # Backward layer 1
    grad_bn2 = grad_relu(relu2, bn2, grad_flat)
    grad_conv2 = grad_batch_norm(conv2, bn2, grad_bn2)
    grad_relu1, grad_filter_2 = grad_conv2d(relu1, filter_2, conv2, grad_conv2, num_filters, num_filters, kernel_size, stride, padding)

    # Backward layer 2
    grad_bn1 = grad_relu(relu1, bn1, grad_relu1)
    grad_conv1 = grad_batch_norm(conv1, bn1, grad_bn1)
    grad_relu3, grad_filter_1 = grad_conv2d(img, filter_1, conv1, grad_conv1, channel_size, num_filters, kernel_size, stride, padding)

    # Reshape to proper size (IMG x 10)
    grad_weight = torch.matmul(flattened.T, grad_loss)  

    return grad_filter_1, grad_filter_2, grad_weight



def unit_test_CNN(img, label, filter_1, filter_2, weight, channel_size=1, num_filters=1, kernel_size=3, stride=1, padding=1):
    # call your implemented "CNN"
    img.requires_grad_()
    filter_1.requires_grad_()
    filter_2.requires_grad_()
    weight.requires_grad_()
    y = CNN(img, filter_1, filter_2, weight, channel_size=channel_size, num_filters=num_filters, kernel_size=kernel_size, stride=stride, padding=padding)
    y.requires_grad_()

    # compute loss function
    loss = F.cross_entropy(y, label).mean()
    loss.requires_grad_()

    # compute gradient of loss w.r.t. logits
    grad_loss = torch.autograd.grad(loss, y, retain_graph=True)[0]

    # call your implemented "grad_batch_norm" function
    grad_filter_1, grad_filter_2, grad_weight = grad_CNN(img, filter_1, filter_2, weight, grad_loss, channel_size=channel_size, num_filters=num_filters, kernel_size=kernel_size, stride=stride, padding=padding)

    # compute ground-truth gradients
    grad_filter_1_gt = torch.autograd.grad(loss, filter_1, retain_graph=True)[0]
    grad_filter_2_gt = torch.autograd.grad(loss, filter_2, retain_graph=True)[0]
    grad_weight_gt = torch.autograd.grad(loss, weight, retain_graph=True)[0]

    diff = (grad_filter_1 - grad_filter_1_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_filter_1 is correct!")
    else:
        print("Your implementation of grad_filter_1 is wrong!")

    diff = (grad_filter_2 - grad_filter_2_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_filter_2 is correct!")
    else:
        print("Your implementation of grad_filter_2 is wrong!")

    diff = (grad_weight - grad_weight_gt).norm()
    if diff < 1.0e-5:
        print("Your implementation of grad_weight is correct!")
    else:
        print("Your implementation of grad_weight is wrong!")


filter_1 = torch.randn(D, C, K, K) # filter shape: D x C x K x K
filter_2 = torch.randn(D, D, K, K) # filter shape: D x C x K x K

### compute the correct shape and then replace None with it ###
#print(img.size())
size = 28
# Size should be D x img H x W 
weight = torch.randn(D*size*size, 10)

unit_test_CNN(img, label, filter_1, filter_2, weight, channel_size=C, num_filters=D, kernel_size=K, stride=1, padding=P)

Your implementation of grad_filter_1 is correct!
Your implementation of grad_filter_2 is correct!
Your implementation of grad_weight is correct!
